In [13]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.model_selection import train_test_split

# --- Configuration (Using your fixed relative paths) ---
PATH_ENRON_CSV = os.path.join("Raw_data", "emails.csv")
PATH_PHISHING_CSV = os.path.join("Raw_data", "phishing_emails.csv")

# --- Helper Function: Text Cleaning ---
def defang_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Remove Email Headers
    if "Message-ID:" in text[:500]:
        if "\n\n" in text:
            text = text.split("\n\n", 1)[1]

    # 2. Remove HTML Tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. Remove common technical jargon/CSS
    text = re.sub(r'\b(font|face|span|style|mso|bidi|class|align)\b', ' ', text, flags=re.IGNORECASE)

    # 4. Basic cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ Setup complete. Cleaning engine ready.")

✅ Setup complete. Cleaning engine ready.


In [14]:
print(f"Loading Enron CSV from: {PATH_ENRON_CSV}")
try:
    # UPGRADE: Force string data type to save memory
    df_normal = pd.read_csv(PATH_ENRON_CSV, dtype=str)
    
    # Robust Column Renaming
    if 'message' in df_normal.columns and 'text' not in df_normal.columns:
        df_normal.rename(columns={'message': 'text'}, inplace=True)
    elif 'content' in df_normal.columns and 'text' not in df_normal.columns:
        df_normal.rename(columns={'content': 'text'}, inplace=True)

    # UPGRADE: Fill NaNs explicitly before processing
    df_normal['text'] = df_normal['text'].fillna('')

    print("Cleaning Enron emails (This might take a minute)...")
    df_normal['text'] = df_normal['text'].apply(defang_text)
    
    # UPGRADE: The Outlier Filter (Keep lengths between 10 and 30,000)
    df_normal = df_normal[(df_normal['text'].str.len() > 10) & (df_normal['text'].str.len() <= 30000)].copy()
    
    df_normal['label'] = 0
    df_normal = df_normal[['text', 'label']]
    print(f"✅ Loaded {len(df_normal)} clean Normal emails.")
except FileNotFoundError:
    print(f"❌ Error: Enron file not found at {PATH_ENRON_CSV}")

Loading Enron CSV from: Raw_data\emails.csv
Cleaning Enron emails (This might take a minute)...
✅ Loaded 513547 clean Normal emails.


In [15]:
print(f"\nLoading Phishing CSV from: {PATH_PHISHING_CSV}")
try:
    df_phishing = pd.read_csv(PATH_PHISHING_CSV, dtype=str)
    
    # Handle Phishing-specific column names
    possible_phish_cols = ['Email Text', 'body', 'text', 'Message']
    for col in possible_phish_cols:
        if col in df_phishing.columns:
            df_phishing.rename(columns={col: 'text'}, inplace=True)
            break
            
    if 'text' not in df_phishing.columns:
        raise KeyError(f"Could not find text column. Columns are: {list(df_phishing.columns)}")

    df_phishing['text'] = df_phishing['text'].fillna('')

    print("Cleaning Phishing emails...")
    df_phishing['text'] = df_phishing['text'].apply(defang_text)
    
    # The Outlier Filter applies here too!
    df_phishing = df_phishing[(df_phishing['text'].str.len() > 10) & (df_phishing['text'].str.len() <= 30000)].copy()
    
    df_phishing['label'] = 1
    df_phishing = df_phishing[['text', 'label']]
    print(f"✅ Loaded {len(df_phishing)} clean Phishing emails.")
except Exception as e:
    print(f"❌ Error loading Phishing data: {e}")


Loading Phishing CSV from: Raw_data\phishing_emails.csv
Cleaning Phishing emails...
✅ Loaded 18041 clean Phishing emails.


In [16]:
# 1. Combine into a master dataset
df_combined = pd.concat([df_normal, df_phishing], ignore_index=True)
output_path = os.path.join("Clean_data", "processed_emails.csv")
df_combined.to_csv(output_path, index=False)
print(f"🚀 Master Dataset saved to: {output_path}")

# 2. Separate the classes for splitting
safe_emails = df_combined[df_combined['label'] == 0]
phishing_emails = df_combined[df_combined['label'] == 1]

# 3. The Split (80% Safe for Train/Val pool, 20% Safe for Battlefield Test)
train_safe, test_safe = train_test_split(safe_emails, test_size=0.2, random_state=42)

# Take 10% of the training pool for Validation
train_safe, val_safe = train_test_split(train_safe, test_size=0.1, random_state=42)

# Combine the held-back Normal emails with ALL the Phishing emails for the Test Set
df_test = pd.concat([test_safe, phishing_emails]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- Stage 3: Final Data Distribution ---")
print(f"✅ Training Set (Normal Only): {len(train_safe)} rows")
print(f"✅ Validation Set (Normal Only): {len(val_safe)} rows")
print(f"✅ Test Set (The Simulation): {len(df_test)} rows")

# 4. Save the splits
train_safe.to_csv(os.path.join("Clean_data", "train_safe.csv"), index=False)
val_safe.to_csv(os.path.join("Clean_data", "val_safe.csv"), index=False)
df_test.to_csv(os.path.join("Clean_data", "test_mixed.csv"), index=False)
print("\n✅ All splits successfully saved to Clean_data folder.")

🚀 Master Dataset saved to: Clean_data\processed_emails.csv

--- Stage 3: Final Data Distribution ---
✅ Training Set (Normal Only): 369753 rows
✅ Validation Set (Normal Only): 41084 rows
✅ Test Set (The Simulation): 120751 rows

✅ All splits successfully saved to Clean_data folder.
